[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burzynski-lab/OU_PHYS2222_ComputationalPhysics/blob/main/LectureNotes/10_ODEs.ipynb)

# Ordinary Differential Equations

**PHYS 2222: Computational Physics** · University of Oklahoma · Spring 2027 · Week 7–8

> **Before you start:** in Colab, choose **File → Save a copy in Drive** so your changes are kept.

## Learning goals

By the end of this notebook you should be able to:

- say what an **initial value problem** is, and rewrite a second-order equation (like Newton's second law) as a system of first-order equations using a **state vector**;
- derive and code **Euler's method**, and explain why its error is $O(h)$ and why it fails to conserve energy;
- fix the energy problem for mechanics with the **Euler–Cromer** method;
- write the **midpoint (RK2)** and **fourth-order Runge–Kutta (RK4)** methods by hand and measure their convergence on a log-log plot;
- use `scipy.integrate.solve_ivp`, including tolerances, `t_eval`, `dense_output`, and **events**;
- check a numerical solution using **conservation laws** (energy, angular momentum) and known limits;
- recognize when an equation is **stiff**.

## 1. What is a differential equation?

Most equations you have solved have a *number* as the unknown: find $x$ such that $x^2 = 2$. In a **differential equation** the unknown is a whole *function*, and the equation tells you how that function's derivative is related to the function itself.

Here is the most important example in physics. Radioactive nuclei (or unstable particles) decay at random, each with the same probability per unit time. So the more you have, the faster their number $N(t)$ drops:

$$
\frac{dN}{dt} = -\frac{N}{\tau},
$$

where $\tau$ is the **mean lifetime**. In words: "the rate of change of $N$ is proportional to $N$, with a minus sign". A **solution** is any function $N(t)$ that makes this true at every time. You can check by differentiating that

$$
N(t) = N_0\, e^{-t/\tau}
$$

works for *any* constant $N_0$: its derivative is $-\frac{1}{\tau}N_0 e^{-t/\tau} = -N/\tau$. So the equation alone has infinitely many solutions. To pick one you must also say where you start, for example "$N = 1000$ at $t = 0$". That fixes $N_0 = 1000$.

A good way to picture a differential equation is as a **slope field**: at every point $(t, N)$ the equation tells you the slope $dN/dt$ a solution passing through that point must have. A solution is a curve that follows the arrows everywhere. The next cell draws this (you don't need to understand every line of plotting code; `ax.quiver` draws arrows).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

tau_demo = 2.0
T_grid, N_grid = np.meshgrid(np.linspace(0, 6, 15), np.linspace(0, 1000, 12))
slope = -N_grid / tau_demo                       # dN/dt at each grid point
# turn each slope into a short arrow (dt, dN) that looks the same length on the plot
aspect = 1000 / 6                                # N-axis range / t-axis range
dt = 0.3 / np.sqrt(1 + (slope / aspect)**2)
dN = slope * dt

fig, ax = plt.subplots()
ax.quiver(T_grid, N_grid, dt, dN, angles="xy", scale_units="xy", scale=1, color="gray", alpha=0.6)
t_line = np.linspace(0, 6, 200)
for N_start in [250, 500, 1000]:
    ax.plot(t_line, N_start * np.exp(-t_line / tau_demo), label=f"start at N = {N_start}")
ax.set_xlabel("t (arbitrary time units)")
ax.set_ylabel("N")
ax.set_title(r"Slope field of $dN/dt = -N/\tau$ and three solutions")
ax.legend()
plt.show()

`np.meshgrid` (which you may not have seen) turns two 1D arrays into 2D grids of $t$ and $N$ values, so we can compute the slope at every grid point at once.

Each starting value gives a different curve, but all of them follow the arrows. This is also the idea behind every numerical method in this notebook: **start at the known point, look up the slope, take a small step along it, and repeat.**

## 2. Initial value problems

An **ordinary differential equation** (ODE) relates an unknown function of one variable (here, time) to its derivatives. The general first-order form is

$$
\frac{dy}{dt} = f(t, y), \qquad y(t_0) = y_0 .
$$

The equation tells you the slope of $y(t)$ at every point; the **initial condition** $y(t_0) = y_0$ tells you where to start. Together they make an **initial value problem** (IVP). If $f$ is well behaved, the IVP has exactly one solution.

Physics is full of these:

- radioactive decay: $dN/dt = -N/\tau$, with $N(0) = N_0$;
- Newton's second law: $m\, d^2x/dt^2 = F(x, v, t)$, with $x(0)$ and $v(0)$ given;
- a charge in electric and magnetic fields, a planet around the Sun, a pendulum, a circuit.

Only a few of these have a solution you can write down with pencil and paper. Add air drag to a baseball, a large amplitude to a pendulum, or a third body to the Kepler problem, and the closed-form solution is gone. A numerical method gives you the solution at a sequence of times $t_0, t_0 + h, t_0 + 2h, \ldots$, where $h$ is the **step size**.

### Second-order equations as first-order systems

The **order** of a differential equation is the highest derivative that appears. The decay equation is first order. Newton's second law, $F = ma$ with $a = d^2x/dt^2$, is **second order**, which is why mechanics needs *two* initial conditions: where the object starts and how fast it is moving.

All the methods in this notebook solve **first-order** equations. That is not a limitation, because any higher-order ODE can be turned into a system of first-order ODEs. Take Newton's second law in one dimension,

$$
\frac{d^2 x}{dt^2} = a(x, v, t), \qquad a = F/m .
$$

Introduce the velocity $v = dx/dt$ as a second unknown. Then

$$
\frac{dx}{dt} = v, \qquad \frac{dv}{dt} = a(x, v, t).
$$

Collect the unknowns into a **state vector** $\mathbf{y} = (x, v)$. The system is now exactly of the form $d\mathbf{y}/dt = \mathbf{f}(t, \mathbf{y})$ with

$$
\mathbf{f}(t, \mathbf{y}) = \big(v,\; a(x, v, t)\big).
$$

In 3D you would use $\mathbf{y} = (x, y, z, v_x, v_y, v_z)$: six first-order equations. The state vector holds everything you need to know to predict the future.

In code, $\mathbf{y}$ is a NumPy array and $\mathbf{f}$ is a Python function `f(t, y)` that returns an array of the same length. This `f(t, y)` convention, with `t` first, is the same one SciPy uses, so every function you write today can be passed straight to `solve_ivp` later.

In [ ]:
from scipy.integrate import solve_ivp
import scipy.constants as const

**Physics recap: the harmonic oscillator.** A mass $m$ on a spring of stiffness $k$ feels Hooke's-law force $F = -kx$, pulling it back toward $x = 0$. Newton's law gives

$$
\ddot x = -\omega^2 x, \qquad \omega = \sqrt{k/m},
$$

where the dots mean time derivatives ($\dot x = dx/dt$, $\ddot x = d^2x/dt^2$). Released from rest at $x = A$, the mass oscillates as $x(t) = A\cos\omega t$ with period $2\pi/\omega$. The total energy, kinetic plus spring potential, $E = \tfrac12 m v^2 + \tfrac12 k x^2$, stays constant. The same equation describes a pendulum at small angles, a vibrating molecule, an LC circuit, and (in quantum mechanics) the vibrations of atoms in a solid, so it is the standard test problem for ODE solvers.

Here is the harmonic oscillator written as a first-order system. The line `x, v = y` **unpacks** the two entries of the array `y` into two named variables, which makes the physics easier to read.

In [ ]:
omega = 2 * np.pi          # angular frequency (rad/s), so the period is 1 s

def sho(t, y):
    """Right-hand side for the harmonic oscillator. y = [x, v]."""
    x, v = y
    return np.array([v, -omega**2 * x])

print(sho(0.0, np.array([1.0, 0.0])))   # at x = 1 m, v = 0: velocity 0, acceleration -omega^2

## 3. Euler's method

Recall the **forward difference** from the differentiation notebook:

$$
\frac{dy}{dt} \approx \frac{y(t + h) - y(t)}{h}, \qquad \text{error } O(h).
$$

Set the left side equal to $f(t, y)$ and solve for $y(t+h)$:

$$
\boxed{\,y_{n+1} = y_n + h\, f(t_n, y_n)\,}
$$

That is **Euler's method**: start at $y_0$, compute the slope, take a straight-line step of length $h$, repeat. From the Taylor series

$$
y(t + h) = y(t) + h\, y'(t) + \tfrac{1}{2} h^2 y''(t) + \cdots
$$

you can see that each step drops the $\tfrac12 h^2 y''$ term, so the **local** error per step is $O(h^2)$. To reach a fixed final time $T$ you take $T/h$ steps, so the errors add up to a **global** error of order $(T/h)\cdot h^2 = O(h)$. Euler's method is a **first-order** method.

The function below works for a scalar equation or a system: `y0` can be a list like `[1.0, 0.0]`. It stores the whole solution in a 2D array with one row per time and one column per component of the state vector.

In [ ]:
def euler(f, y0, t):
    """Solve dy/dt = f(t, y) with Euler's method on the time grid t.

    Returns an array of shape (len(t), len(y0)); row n is y at time t[n].
    """
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for n in range(len(t) - 1):
        h = t[n + 1] - t[n]
        y[n + 1] = y[n] + h * f(t[n], y[n])
    return y

### Example: muon decay

Muons are heavier cousins of the electron, produced when cosmic rays hit the upper atmosphere. They decay with mean lifetime $\tau = 2.197\ \mu\text{s}$ (measured in their rest frame). The number remaining obeys the decay equation from Section 1,

$$
\frac{dN}{dt} = -\frac{N}{\tau}, \qquad N(t) = N_0\, e^{-t/\tau}.
$$

We work in microseconds, so $\tau = 2.197$. Even a one-component equation must return an array, so `decay` returns `np.array([...])` and $N$ is `y[0]`.

In [ ]:
tau = 2.197                     # muon lifetime (microseconds)
N0 = 1000.0

def decay(t, y):
    return np.array([-y[0] / tau])

h = 0.5                                  # step size (microseconds)
t = np.arange(0, 10 + h / 2, h)          # 0, 0.5, ..., 10 (the h/2 makes sure 10 is included)
N_euler = euler(decay, [N0], t)[:, 0]    # [:, 0] picks out the single column
N_exact = N0 * np.exp(-t / tau)

fig, ax = plt.subplots()
ax.plot(t, N_exact, label="exact")
ax.plot(t, N_euler, "o", label=f"Euler, h = {h} μs")
ax.set_xlabel("t (μs)")
ax.set_ylabel("muons remaining N")
ax.legend()
plt.show()

In [ ]:
rel_err = abs(N_euler[-1] - N_exact[-1]) / N_exact[-1]
print(f"N(10 μs): Euler = {N_euler[-1]:.3f}, exact = {N_exact[-1]:.3f}, relative error = {rel_err:.1%}")

Euler undershoots: the slope is steepest at the start of each step, so every straight-line step overshoots downward. With $h = 0.5\ \mu$s (about $\tau/4$) the error at the end is tens of percent. Let's measure how the error depends on $h$.

### Error versus step size

We repeat the calculation for a range of step sizes and plot the error at $t = 10\ \mu$s on log-log axes. A straight line of slope $p$ on a log-log plot means error $\propto h^p$.

In [ ]:
T = 10.0
hs = np.array([0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001])
errors = []
for h in hs:
    t = np.linspace(0, T, int(round(T / h)) + 1)
    N_end = euler(decay, [N0], t)[-1, 0]
    errors.append(abs(N_end - N0 * np.exp(-T / tau)))

fig, ax = plt.subplots()
ax.loglog(hs, errors, "o-", label="Euler")
ax.loglog(hs, errors[-1] * hs / hs[-1], "k--", label=r"$\propto h$")
ax.set_xlabel("step size h (μs)")
ax.set_ylabel("|N(10 μs) − exact|")
ax.legend()
plt.show()

The points follow the slope-1 line: halving $h$ halves the error. To get one more correct digit you need ten times as many steps. That is expensive, and for long simulations the cost adds up fast. We will do much better shortly.

### Your turn 1

A two-step decay chain $A \to B \to C$ ($C$ stable) obeys

$$
\frac{dN_A}{dt} = -\frac{N_A}{\tau_A}, \qquad \frac{dN_B}{dt} = \frac{N_A}{\tau_A} - \frac{N_B}{\tau_B}.
$$

Write a right-hand-side function for the state vector $(N_A, N_B)$ and solve it with `euler` for $\tau_A = 2$ and $\tau_B = 1$ (any time unit), starting from $N_A = 1000$, $N_B = 0$, out to $t = 10$. Plot both populations, and compare $N_B(t)$ to the exact result

$$
N_B(t) = N_A(0)\,\frac{\tau_B}{\tau_A - \tau_B}\left(e^{-t/\tau_A} - e^{-t/\tau_B}\right).
$$

In [ ]:
# Your code here

## 4. Euler's method and energy drift

Now apply Euler to the harmonic oscillator `sho` from Section 2, with $\omega = 2\pi$ rad/s (period 1 s), starting at $x = 1$ m at rest. The exact solution is $x(t) = \cos\omega t$. Per unit mass, the energy

$$
E = \tfrac12 v^2 + \tfrac12 \omega^2 x^2
$$

should stay at its initial value $\tfrac12\omega^2$ forever.

In [ ]:
h = 0.01
t = np.arange(0, 10 + h / 2, h)
y = euler(sho, [1.0, 0.0], t)
x, v = y[:, 0], y[:, 1]
E = 0.5 * v**2 + 0.5 * omega**2 * x**2

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(7, 6))
ax1.plot(t, np.cos(omega * t), label="exact")
ax1.plot(t, x, label="Euler")
ax1.set_ylabel("x (m)")
ax1.legend()
ax2.plot(t, E / E[0])
ax2.set_xlabel("t (s)")
ax2.set_ylabel("E / E(0)")
plt.show()

The amplitude grows and the energy climbs steadily: Euler's method is pumping energy into a system that has no energy source. Plot the trajectory in **phase space** ($x$ versus $v$): the exact orbit is a closed ellipse, but Euler spirals outward.

In [ ]:
fig, ax = plt.subplots()
ax.plot(x, v, lw=0.8)
ax.set_xlabel("x (m)")
ax.set_ylabel("v (m/s)")
ax.set_title("Euler: phase-space trajectory spirals outward")
ax.set_aspect(1 / omega)    # makes the exact ellipse look like a circle
plt.show()

You can see exactly why. One Euler step is

$$
x_{n+1} = x_n + h v_n, \qquad v_{n+1} = v_n - h\omega^2 x_n .
$$

Square and add (multiply the first by $\omega^2$); the cross terms cancel and you get

$$
\omega^2 x_{n+1}^2 + v_{n+1}^2 = (1 + h^2\omega^2)\,(\omega^2 x_n^2 + v_n^2),
$$

so **every step multiplies the energy by $1 + h^2\omega^2$**, no matter how small $h$ is. Let's check that prediction.

In [ ]:
n_steps = len(t) - 1
print("E_final / E_0 from the simulation:", E[-1] / E[0])
print("(1 + h^2 omega^2)^n_steps          :", (1 + h**2 * omega**2) ** n_steps)

They agree to rounding error. A smaller $h$ slows the drift but never removes it. For a planet orbiting for millions of steps, Euler's method is useless.

## 5. The Euler–Cromer method

A tiny change fixes the drift. Update the velocity first, then use the **new** velocity to update the position:

$$
v_{n+1} = v_n + h\, a(x_n), \qquad x_{n+1} = x_n + h\, v_{n+1}.
$$

This is the **Euler–Cromer** (or *semi-implicit Euler*) method. It is still first order, but it belongs to a family called **symplectic** integrators: they respect the geometry of Hamiltonian mechanics, so the energy oscillates slightly around the true value instead of drifting away. It is written for equations of the form $\ddot x = a(x)$, so we pass the acceleration function instead of the full state-vector function.

In [ ]:
def euler_cromer(accel, x0, v0, t):
    """Euler-Cromer for x'' = accel(x). Returns arrays x, v."""
    x = np.zeros(len(t))
    v = np.zeros(len(t))
    x[0], v[0] = x0, v0
    for n in range(len(t) - 1):
        h = t[n + 1] - t[n]
        v[n + 1] = v[n] + h * accel(x[n])
        x[n + 1] = x[n] + h * v[n + 1]      # note: the NEW velocity
    return x, v

x_ec, v_ec = euler_cromer(lambda x: -omega**2 * x, 1.0, 0.0, t)
E_ec = 0.5 * v_ec**2 + 0.5 * omega**2 * x_ec**2

`lambda x: -omega**2 * x` is a **lambda function**: a one-line function with no name. It is equivalent to writing `def accel(x): return -omega**2 * x` and passing `accel`. Lambdas are handy for short functions you only use once.

In [ ]:
fig, ax = plt.subplots()
ax.plot(t, E / E[0], label="Euler")
ax.plot(t, E_ec / E_ec[0], label="Euler–Cromer")
ax.set_xlabel("t (s)")
ax.set_ylabel("E / E(0)")
ax.legend()
plt.show()
print("Euler-Cromer: max |E/E0 - 1| over 10 periods =", np.max(np.abs(E_ec / E_ec[0] - 1)))

The Euler–Cromer energy wiggles by a few percent but stays bounded for any number of periods. The *phase* of the oscillation is still only first-order accurate, so for precision we want a higher-order method.

## 6. Higher-order methods: midpoint (RK2) and RK4

Euler uses the slope at the *start* of the step. A better idea: use the slope at the *middle* of the step. We don't know $y$ at the midpoint, but we can estimate it with half an Euler step:

$$
k_1 = h\, f(t_n, y_n), \qquad
k_2 = h\, f\!\left(t_n + \tfrac{h}{2},\; y_n + \tfrac{k_1}{2}\right), \qquad
y_{n+1} = y_n + k_2 .
$$

This is the **midpoint method**, also called second-order Runge–Kutta (**RK2**). Expanding in a Taylor series shows that the $h^2$ terms now match the true solution, so the local error is $O(h^3)$ and the global error is $O(h^2)$. It costs two evaluations of $f$ per step.

Carry the idea further, combining four slope estimates with carefully chosen weights, and you get the classic **fourth-order Runge–Kutta** method (**RK4**):

$$
\begin{aligned}
k_1 &= h\, f(t_n, y_n), \\
k_2 &= h\, f\!\left(t_n + \tfrac{h}{2},\; y_n + \tfrac{k_1}{2}\right), \\
k_3 &= h\, f\!\left(t_n + \tfrac{h}{2},\; y_n + \tfrac{k_2}{2}\right), \\
k_4 &= h\, f(t_n + h,\; y_n + k_3), \\
y_{n+1} &= y_n + \tfrac{1}{6}\,(k_1 + 2k_2 + 2k_3 + k_4).
\end{aligned}
$$

Its global error is $O(h^4)$: halve the step and the error drops by $2^4 = 16$. RK4 has been the workhorse of computational physics for decades. (Newman's *Computational Physics*, Section 8.1, derives these formulas.)

To compare the methods fairly, we write each one as a **step function** that advances the solution by one step, and a single driver that loops over the time grid.

In [ ]:
def euler_step(f, t, y, h):
    return y + h * f(t, y)

def rk2_step(f, t, y, h):
    k1 = h * f(t, y)
    k2 = h * f(t + h / 2, y + k1 / 2)
    return y + k2

def rk4_step(f, t, y, h):
    k1 = h * f(t, y)
    k2 = h * f(t + h / 2, y + k1 / 2)
    k3 = h * f(t + h / 2, y + k2 / 2)
    k4 = h * f(t + h, y + k3)
    return y + (k1 + 2 * k2 + 2 * k3 + k4) / 6

The driver takes the step function as an argument. In Python, functions are objects like any other, so you can pass `rk4_step` to `odesolve` just as you would pass a number.

In [ ]:
def odesolve(step, f, y0, t):
    """Solve dy/dt = f(t, y) on the grid t using the one-step method `step`."""
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for n in range(len(t) - 1):
        y[n + 1] = step(f, t[n], y[n], t[n + 1] - t[n])
    return y

h = 0.05
t = np.arange(0, 20 + h / 2, h)          # 20 periods
for name, step in [("Euler", euler_step), ("RK2", rk2_step), ("RK4", rk4_step)]:
    y = odesolve(step, sho, [1.0, 0.0], t)
    E_end = 0.5 * y[-1, 1]**2 + 0.5 * omega**2 * y[-1, 0]**2
    print(f"{name:5s}: x(20 s) = {y[-1, 0]: .6f} (exact 1),  E_end/E0 = {E_end / (0.5 * omega**2):.6f}")

With the same step size (20 steps per period), Euler has blown up completely, RK2 has drifted badly in both phase and energy, and RK4 is still within about 0.3%.

### Convergence comparison

Now the key figure: error at a fixed time versus $h$ for all three methods, on log-log axes.

In [ ]:
T = 2.0
hs = 2.0 ** -np.arange(3, 13)        # 1/8 down to 1/4096 s
err = {"Euler": [], "RK2": [], "RK4": []}
steps = {"Euler": euler_step, "RK2": rk2_step, "RK4": rk4_step}
for h in hs:
    t = np.linspace(0, T, int(round(T / h)) + 1)
    for name in err:
        x_end = odesolve(steps[name], sho, [1.0, 0.0], t)[-1, 0]
        err[name].append(abs(x_end - np.cos(omega * T)))

fig, ax = plt.subplots()
for name, p in [("Euler", 1), ("RK2", 2), ("RK4", 4)]:
    line, = ax.loglog(hs, err[name], "o-", label=name)
    ax.loglog(hs, err[name][2] * (hs / hs[2])**p, "--", color=line.get_color(), lw=0.8)
ax.set_xlabel("step size h (s)")
ax.set_ylabel("|x(T) − exact| (m)")
ax.set_title("dashed lines: slopes 1, 2, 4")
ax.legend()
plt.show()

How to read this plot:

- Each method follows its predicted slope: 1 for Euler, 2 for RK2, 4 for RK4.
- At $h \approx 0.01$ s, RK4 is already accurate to $10^{-7}$ m or better while Euler is off by order one. RK4 costs 4 function evaluations per step instead of 1, but for the same accuracy it needs far fewer steps.
- At the very smallest $h$, the RK4 curve flattens out or turns up. That is **round-off error** accumulating over many steps, just like the optimal-$h$ trade-off you saw for numerical derivatives.
- The line `line, = ax.loglog(...)` unpacks the list of lines that `loglog` returns, so we can reuse each method's color for its dashed reference line.

### Your turn 2

RK4 is very accurate, but it is **not** symplectic. Run the harmonic oscillator for 1000 periods with $h = 0.05$ s using both `rk4_step` (via `odesolve`) and `euler_cromer`. Plot $E/E_0 - 1$ versus time for both. Which one drifts? Which one is more accurate after a few periods? What does that suggest about choosing a method for a long-term planetary simulation?

In [ ]:
# Your code here

## 7. `scipy.integrate.solve_ivp`

In practice you usually use a library solver. SciPy's `solve_ivp` implements several methods; the default, `'RK45'`, is a Runge–Kutta pair that computes a 4th- and a 5th-order step together. Their difference estimates the error, and the solver uses it to **choose the step size automatically**: small steps where the solution changes quickly, big steps where it is smooth.

The basic call is

```python
sol = solve_ivp(f, (t_start, t_end), y0)
```

where `f(t, y)` is the same right-hand side you have been writing. The result is an object with attributes:

| attribute | meaning |
|---|---|
| `sol.t` | the times at which the solution was computed (1D array) |
| `sol.y` | the solution, shape `(len(y0), len(sol.t))`: **one row per variable** |
| `sol.success`, `sol.message` | whether it worked, and why not |
| `sol.nfev` | how many times `f` was called |

Watch out: `sol.y` is the **transpose** of the arrays our own `odesolve` returns. `sol.y[0]` is $x(t)$, `sol.y[1]` is $v(t)$.

In [ ]:
sol = solve_ivp(sho, (0, 10), [1.0, 0.0])
print(sol.message)
print("number of time points:", sol.t.size, "  shape of sol.y:", sol.y.shape, "  f calls:", sol.nfev)
print("first few step sizes:", np.diff(sol.t)[:5])

### Tolerances and `t_eval`

`solve_ivp` adjusts the step so that the estimated error in each step is below `atol + rtol * |y|`. The **defaults are loose**: `rtol=1e-3`, `atol=1e-6`. For physics you should almost always set them yourself.

By default you only get the solution at the solver's own (irregular) time steps. Pass `t_eval=` an array of times to get the solution exactly there, for example for a smooth plot.

In [ ]:
t_plot = np.linspace(0, 10, 1001)
fig, ax = plt.subplots()
for rtol in [1e-3, 1e-6, 1e-10]:
    sol = solve_ivp(sho, (0, 10), [1.0, 0.0], t_eval=t_plot, rtol=rtol, atol=rtol * 1e-3)
    ax.semilogy(sol.t, np.abs(sol.y[0] - np.cos(omega * sol.t)) + 1e-17,
                label=f"rtol = {rtol:g} ({sol.nfev} calls)")
ax.set_xlabel("t (s)")
ax.set_ylabel("|x − exact| (m)")
ax.legend()
plt.show()

With the default tolerance the error after 10 periods is about 2%. Tightening `rtol` by a factor of $10^7$ makes the error 10 million times smaller but costs only about 30 times more function calls. (The `+ 1e-17` just avoids taking the log of an exact zero at $t=0$.)

### Dense output

With `dense_output=True`, the solver also returns an interpolating function `sol.sol(t)` that gives the solution at **any** time in the interval, at the accuracy of the solver, without re-solving.

In [ ]:
sol = solve_ivp(sho, (0, 10), [1.0, 0.0], rtol=1e-10, atol=1e-12, dense_output=True)
for tq in [0.25, 3.1, 7.77]:
    print(f"x({tq}) = {sol.sol(tq)[0]: .10f}   exact {np.cos(omega * tq): .10f}")

### Passing parameters with `args`

Often the right-hand side depends on parameters (a mass, a drag coefficient). Add them as extra arguments after `t, y` and pass their values with `args=(...)`. Note that `args` must be a **tuple**, so a single parameter is written `args=(value,)` with a trailing comma.

In [ ]:
def sho_w(t, y, w):
    x, v = y
    return np.array([v, -w**2 * x])

sol = solve_ivp(sho_w, (0, 1), [1.0, 0.0], args=(4 * np.pi,), rtol=1e-10, atol=1e-12)
print("x(1 s) with omega = 4 pi:", sol.y[0, -1], "(two full periods, so expect 1)")

### Events

Often you want to stop when something happens: a ball hits the ground, a pendulum passes through the bottom. An **event function** `event(t, y)` returns a number that crosses zero when the event happens. `solve_ivp` locates the crossing accurately (by root finding, like you did with `brentq`). Two optional attributes control it:

- `event.terminal = True` stops the integration at the event;
- `event.direction = -1` triggers only when the function goes from positive to negative (`+1` for negative to positive, `0`, the default, for both).

The times of the events come back in `sol.t_events` and the states in `sol.y_events`, one list per event function. We use events in the next three examples.

## 8. Projectile with air drag

For a fast, baseball-sized object, air drag is well described by the **quadratic** drag force

$$
\mathbf{F}_d = -\tfrac12 \rho\, C_d A\, |\mathbf{v}|\,\mathbf{v},
$$

where $\rho$ is the air density, $A$ the cross-sectional area, and $C_d$ a dimensionless drag coefficient. With $b = \rho C_d A/(2m)$, the equations of motion in the $x$–$z$ plane are

$$
\dot x = v_x,\quad \dot z = v_z,\quad \dot v_x = -b\,|\mathbf v|\,v_x,\quad \dot v_z = -g - b\,|\mathbf v|\,v_z .
$$

This has no closed-form solution, because $|\mathbf v| = \sqrt{v_x^2 + v_z^2}$ couples the two directions. We use typical values for a baseball: $m = 0.145$ kg, diameter 7.4 cm, $C_d \approx 0.35$, $\rho = 1.2\ \text{kg/m}^3$, launched at 45 m/s (about 100 mph).

In [ ]:
g = const.g                         # 9.80665 m/s^2
m_ball = 0.145                      # kg
A = np.pi * (0.074 / 2)**2          # m^2
b_ball = 1.2 * 0.35 * A / (2 * m_ball)   # 1/m
print(f"b = {b_ball:.4f} 1/m;  terminal speed sqrt(g/b) = {np.sqrt(g / b_ball):.1f} m/s")

def projectile(t, y, b):
    x, z, vx, vz = y
    speed = np.sqrt(vx**2 + vz**2)
    return np.array([vx, vz, -b * speed * vx, -g - b * speed * vz])

def hit_ground(t, y, b):         # event functions get the same arguments as f
    return y[1]                  # height z; the event is z = 0
hit_ground.terminal = True
hit_ground.direction = -1        # only when falling

Setting `hit_ground.terminal = True` attaches an attribute to the function object; `solve_ivp` looks for it. We give the solver a generous time span (20 s) and let the event stop it.

In [ ]:
v0, angle = 45.0, np.radians(35)
y0 = [0.0, 0.0, v0 * np.cos(angle), v0 * np.sin(angle)]

fig, ax = plt.subplots()
for b, label in [(0.0, "vacuum"), (b_ball, "quadratic drag")]:
    sol = solve_ivp(projectile, (0, 20), y0, args=(b,), events=hit_ground,
                    rtol=1e-9, atol=1e-9, dense_output=True)
    t_land = sol.t_events[0][0]
    x_land = sol.y_events[0][0][0]
    tt = np.linspace(0, t_land, 300)
    xx, zz = sol.sol(tt)[:2]
    ax.plot(xx, zz, label=f"{label}: range {x_land:.1f} m, flight time {t_land:.2f} s")

ax.set_xlabel("x (m)")
ax.set_ylabel("height z (m)")
ax.set_aspect("equal")
ax.legend(fontsize=9)
plt.show()
print("vacuum range from the formula v0^2 sin(2 theta)/g =", v0**2 * np.sin(2 * angle) / g, "m")

The vacuum run reproduces $R = v_0^2 \sin 2\theta / g$ to many digits, which is a good check that the code is right. With drag the range drops from 194 m to about 110 m, and the trajectory is no longer a symmetric parabola: the ball comes down more steeply than it went up.

### Your turn 3

With drag, is 45° still the launch angle that gives the longest range? Loop over angles from 20° to 60° in 1° steps, compute the range with drag for each (same $v_0$ and $b$), plot range versus angle for both the vacuum and drag cases, and print the best angle with drag.

In [ ]:
# Your code here

## 9. The nonlinear pendulum

**Physics recap.** A simple pendulum is a mass on a light rod of length $L$. Gravity's component along the arc of motion is $-mg\sin\theta$, and the distance along the arc is $s = L\theta$, so Newton's law $m\ddot s = -mg\sin\theta$ becomes

$$
\ddot\theta = -\frac{g}{L}\sin\theta .
$$

For small angles, $\sin\theta \approx \theta$ gives simple harmonic motion with period $T_0 = 2\pi\sqrt{L/g}$, independent of amplitude. For large amplitudes the period gets longer. The exact result is an elliptic integral,

$$
T = 4\sqrt{\frac{L}{g}}\; K\!\left(\sin^2\frac{\theta_0}{2}\right),
$$

with $K(m)$ the complete elliptic integral of the first kind (`scipy.special.ellipk` uses this same $m$ convention). We will measure the period from the ODE using an event and compare.

Released from rest at $\theta_0 > 0$, the pendulum reaches the bottom ($\theta = 0$, moving in the negative direction) after exactly a quarter period. So a terminal event at $\theta = 0$ with `direction = -1` gives $T = 4\,t_{\text{event}}$.

In [ ]:
from scipy.special import ellipk

L_pend = 1.0                           # m
T0 = 2 * np.pi * np.sqrt(L_pend / g)   # small-angle period (s)

def pendulum(t, y):
    theta, w = y
    return np.array([w, -(g / L_pend) * np.sin(theta)])

def at_bottom(t, y):
    return y[0]
at_bottom.terminal = True
at_bottom.direction = -1

def period(theta0):
    sol = solve_ivp(pendulum, (0, 10 * T0), [theta0, 0.0], events=at_bottom, rtol=1e-10, atol=1e-12)
    return 4 * sol.t_events[0][0]

for deg in [5, 30, 90, 150, 179]:
    th = np.radians(deg)
    T_exact = 4 * np.sqrt(L_pend / g) * ellipk(np.sin(th / 2)**2)
    print(f"theta0 = {deg:3d} deg: T/T0 = {period(th) / T0:.6f}   elliptic integral: {T_exact / T0:.6f}")

In [ ]:
amps = np.radians(np.linspace(1, 175, 60))
T_num = np.array([period(a) for a in amps])     # a "list comprehension" turned into an array

fig, ax = plt.subplots()
ax.plot(np.degrees(amps), T_num / T0, label="numerical (solve_ivp)")
ax.axhline(1, color="k", ls="--", label="small-angle approximation")
ax.set_xlabel(r"amplitude $\theta_0$ (degrees)")
ax.set_ylabel(r"period $T / T_0$")
ax.legend()
plt.show()

The line `[period(a) for a in amps]` is a **list comprehension**: a compact way to build a list by applying an expression to each element of another sequence. It does the same as a `for` loop with `.append()`.

The small-angle formula is good to better than 1% up to about 20°. Near 180° (balanced upside down) the period diverges: the pendulum lingers near the top. The time series makes the difference visible.

In [ ]:
t_plot = np.linspace(0, 4 * T0, 800)
fig, ax = plt.subplots()
for deg in [10, 150]:
    th0 = np.radians(deg)
    sol = solve_ivp(pendulum, (0, 4 * T0), [th0, 0.0], t_eval=t_plot, rtol=1e-10, atol=1e-12)
    line, = ax.plot(t_plot / T0, np.degrees(sol.y[0]), label=f"{deg}° (full equation)")
    ax.plot(t_plot / T0, deg * np.cos(2 * np.pi * t_plot / T0), "--", color=line.get_color(),
            lw=0.8, label=f"{deg}° (small-angle formula)")
ax.set_xlabel(r"$t / T_0$")
ax.set_ylabel(r"$\theta$ (degrees)")
ax.legend(fontsize=8, loc="lower left")
plt.show()

## 10. A Kepler orbit, with conservation checks

**Physics recap.** Newton's law of gravity says the Sun (mass $M_\odot$) pulls a planet of mass $m$ toward it with force of magnitude $GM_\odot m/r^2$, where $r$ is the distance between them. Dividing by $m$ and pointing the force toward the Sun at the origin, a planet (or comet) obeys

$$
\ddot{\mathbf r} = -\frac{GM_\odot}{r^3}\,\mathbf r .
$$

(Dividing $\mathbf r$ by $r^3$ gives a vector of length $1/r^2$ pointing away from the Sun; the minus sign turns it around.) Kepler discovered, and Newton explained, that the bound orbits are **ellipses** with the Sun at one focus. An ellipse is described by its **semi-major axis** $a$ (half its longest diameter) and **eccentricity** $e$ ($e = 0$ is a circle, $e \to 1$ is a very elongated ellipse). The closest point to the Sun, at distance $a(1-e)$, is the **perihelion**. **Kepler's third law** says the period is $T \propto a^{3/2}$.

SI units would give numbers like $10^{11}$ m and $10^{30}$ kg. Instead use **astronomical units** (AU) for distance and **years** for time. For a circular orbit, gravity supplies the centripetal force, $v^2/r = GM_\odot/r^2$, so $GM_\odot = v^2 r$. Earth moves $2\pi$ AU per year at $r = 1$ AU, which gives $GM_\odot = 4\pi^2\ \text{AU}^3/\text{yr}^2$, and then Kepler's third law reads simply $T = a^{3/2}$ (years, AU). Choosing units so the numbers are of order one is a habit worth building.

The state vector is $(x, y, v_x, v_y)$. We start an orbit with semi-major axis $a = 1$ AU and eccentricity $e = 0.6$ at perihelion, $r = a(1 - e)$, with speed $v = \sqrt{GM(1+e)/(a(1-e))}$.

Two conserved quantities check the solution (both per unit mass):

$$
E = \tfrac12 v^2 - \frac{GM}{r} = -\frac{GM}{2a}, \qquad L_z = x v_y - y v_x .
$$

In [ ]:
GM = 4 * np.pi**2           # AU^3 / yr^2

def kepler(t, s):
    x, y, vx, vy = s
    r3 = (x**2 + y**2) ** 1.5
    return np.array([vx, vy, -GM * x / r3, -GM * y / r3])

a, e = 1.0, 0.6
s0 = [a * (1 - e), 0.0, 0.0, np.sqrt(GM * (1 + e) / (a * (1 - e)))]

def energy(s):
    x, y, vx, vy = s
    return 0.5 * (vx**2 + vy**2) - GM / np.sqrt(x**2 + y**2)

def ang_mom(s):
    x, y, vx, vy = s
    return x * vy - y * vx

print("E(0) =", energy(s0), "  expected -GM/(2a) =", -GM / (2 * a))

`energy` and `ang_mom` work on a single state *or* on the whole `sol.y` array at once, because unpacking `x, y, vx, vy = s` splits a 2D array along its first axis (rows). That is another reason `solve_ivp` puts one variable per row.

Now integrate for 10 orbits with the default and a tight tolerance and compare the conservation laws.

In [ ]:
t_end = 10.0      # years
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
for rtol in [1e-3, 1e-10]:
    sol = solve_ivp(kepler, (0, t_end), s0, rtol=rtol, atol=rtol * 1e-2,
                    t_eval=np.linspace(0, t_end, 4000))
    ax1.plot(sol.y[0], sol.y[1], lw=0.7, label=f"rtol = {rtol:g}")
    dE = np.abs(energy(sol.y) / energy(s0) - 1)
    dL = np.abs(ang_mom(sol.y) / ang_mom(s0) - 1)
    ax2.semilogy(sol.t, dE + 1e-16, label=f"energy, rtol = {rtol:g}")
    ax2.semilogy(sol.t, dL + 1e-16, "--", label=f"ang. mom., rtol = {rtol:g}")
ax1.plot(0, 0, "y*", ms=15)
ax1.set_xlabel("x (AU)")
ax1.set_ylabel("y (AU)")
ax1.set_aspect("equal")
ax1.legend(fontsize=8)
ax2.set_xlabel("t (yr)")
ax2.set_ylabel("relative change")
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

With the default tolerance the orbit visibly precesses and shrinks, and the energy drifts by about 10%: the solver is losing accuracy at every close passage of the Sun, where the planet moves fastest. With `rtol = 1e-10` the orbit closes on itself, and energy and angular momentum stay constant to about $10^{-9}$ or better. **Always check a conserved quantity** when you have one: it is the cheapest test that your simulation is trustworthy.

We can also measure the orbital period with a non-terminal event: the planet crosses the $x$-axis going upward ($y$ from negative to positive) once per orbit, at perihelion.

In [ ]:
def crosses_axis(t, s):
    return s[1]
crosses_axis.direction = 1     # y increasing; not terminal, so all crossings are recorded

sol = solve_ivp(kepler, (0, t_end), s0, events=crosses_axis, rtol=1e-10, atol=1e-12)
t_cross = sol.t_events[0]
t_cross = t_cross[t_cross > 0.01]       # drop a possible detection at the starting point
print("perihelion times (yr):", np.round(t_cross, 6))
print("mean period:", np.mean(np.diff(t_cross)), "yr;  Kepler's third law a^(3/2) =", a**1.5, "yr")

`t_cross > 0.01` is a boolean mask (from the NumPy notebook): it keeps only the times after the start, since the solver may report the starting point $y = 0$ as a crossing.

### Your turn 4

Check **Kepler's third law**, $T^2 \propto a^3$. For semi-major axes $a = 0.4, 0.7, 1, 1.5, 3, 5$ AU (keep $e = 0.6$), compute the period with the event method above, and plot $T$ versus $a$ on log-log axes. Fit the slope with `np.polyfit(np.log(a_values), np.log(T_values), 1)` and compare to $3/2$. Hint: integrate for a few periods' worth of time, which you can estimate in advance.

In [ ]:
# Your code here

## 11. Optional: relativistic motion in a constant electric field

Here is a direct tie to special relativity. An electron in a uniform electric field $E$ feels a constant force $qE$. Newton's second law in its correct relativistic form is about **momentum**:

$$
\frac{dp}{dt} = qE, \qquad \frac{dx}{dt} = v = \frac{p}{\gamma m} = \frac{p/m}{\sqrt{1 + (p/mc)^2}} .
$$

Classically, $v = qEt/m$ grows without limit and passes $c$. Relativistically, $v$ must approach $c$ but never reach it.

SI values (momenta near $10^{-22}$ kg·m/s) would clash with `solve_ivp`'s absolute tolerance, so we pick natural units for this problem: measure momentum in units of $mc$, time in units of $t_0 = mc/(qE)$, and distance in units of $\ell_0 = c\,t_0 = mc^2/(qE)$. With $u = p/mc$, $\tilde t = t/t_0$, $\tilde x = x/\ell_0$, the equations become

$$
\frac{du}{d\tilde t} = 1, \qquad \frac{d\tilde x}{d\tilde t} = \frac{u}{\sqrt{1+u^2}} .
$$

All the physics now sits in the two scale factors. For an electron in a field of 1 MV/m, $\ell_0 = 0.511\ \text{MeV}/(e \cdot 1\ \text{MV/m}) = 0.511$ m.

In [ ]:
E_field = 1e6                                    # V/m
t0 = const.m_e * const.c / (const.e * E_field)  # s
l0 = const.c * t0                                # m
print(f"time unit t0 = {t0 * 1e9:.3f} ns,  length unit l0 = {l0:.4f} m")

def charge_in_field(t, s):
    xt, u = s
    return np.array([u / np.sqrt(1 + u**2), 1.0])

tt = np.linspace(0, 5, 200)
sol = solve_ivp(charge_in_field, (0, 5), [0.0, 0.0], t_eval=tt, rtol=1e-10, atol=1e-12)
u = sol.y[1]
beta_rel = u / np.sqrt(1 + u**2)
beta_classical = tt                  # v/c = qEt/(mc) = t/t0 in these units

In [ ]:
fig, ax = plt.subplots()
ax.plot(tt * t0 * 1e9, beta_rel, label="relativistic")
ax.plot(tt * t0 * 1e9, beta_classical, "--", label="classical (Newtonian)")
ax.axhline(1, color="k", lw=0.8)
ax.set_ylim(0, 2)
ax.set_xlabel("t (ns)")
ax.set_ylabel("v / c")
ax.set_title("Electron in a 1 MV/m field")
ax.legend()
plt.show()

x_exact = np.sqrt(1 + tt**2) - 1          # exact solution, in units of l0
print("max |x_numerical - x_exact| (m):", np.max(np.abs(sol.y[0] - x_exact)) * l0)

The velocity saturates at $c$, while the momentum keeps growing linearly. The exact position is $x = \ell_0\left(\sqrt{1 + (t/t_0)^2} - 1\right)$ (hyperbolic motion), and the numerical solution matches it.

### Your turn 5

Using `charge_in_field` and a **terminal event**, find the time (in ns) and distance (in m) it takes an electron starting from rest in a 1 MV/m field to reach $v = 0.99c$. Then do the same for a proton (`const.m_p`). Remember that $t_0$ and $\ell_0$ depend on the mass, but the dimensionless equations do not. Does the classical formula $t = mv/(qE)$ underestimate or overestimate the time?

In [ ]:
# Your code here

## 12. A note on stiffness

Some equations contain processes on very different time scales: a fast one that dies away quickly and a slow one you actually care about. Chemical reactions, nuclear decay chains with very different lifetimes, and circuits are common examples. Such equations are called **stiff**. For example,

$$
\frac{dy}{dt} = -k\,\big(y - \cos t\big), \qquad k = 1000,
$$

has a solution that relaxes onto $y \approx \cos t$ within a time $1/k$ and then follows it slowly. An explicit method like Euler or RK4 is only **stable** if $h \lesssim 1/k$ (for Euler, $h < 2/k$), even long after the fast part has died away. Exceed it and the numerical solution explodes, no matter how smooth the true solution is.

In [ ]:
k_stiff = 1000.0
def stiff(t, y):
    return np.array([-k_stiff * (y[0] - np.cos(t))])

for h in [0.0019, 0.0021]:
    t = np.arange(0, 1 + h / 2, h)
    y = odesolve(euler_step, stiff, [0.0], t)
    print(f"Euler with h = {h}: y(1) = {y[-1, 0]:.4g}   (true value ~ cos(1) = {np.cos(1):.4f})")

Just above $h = 2/k = 0.002$, Euler explodes. Adaptive solvers like `RK45` survive by shrinking the step, but then they need a huge number of steps. **Implicit** methods (which solve an equation for $y_{n+1}$ at each step) stay stable with large steps. In `solve_ivp`, use `method='Radau'`, `'BDF'`, or `'LSODA'` for stiff problems. Compare the work:

In [ ]:
for method in ["RK45", "Radau", "LSODA"]:
    sol = solve_ivp(stiff, (0, 10), [0.0], method=method, rtol=1e-6, atol=1e-9)
    print(f"{method:6s}: {sol.nfev:6d} function calls, y(10) = {sol.y[0, -1]:.6f}")

If `solve_ivp` with the default method seems to take forever, or `nfev` is enormous, suspect stiffness and try `'LSODA'` (which switches automatically) or `'Radau'`.

## Summary

- An **initial value problem** is $d\mathbf y/dt = \mathbf f(t, \mathbf y)$ with $\mathbf y(t_0)$ given. Rewrite higher-order equations as first-order systems with a **state vector**, e.g. $(x, v)$.
- **Euler:** $y_{n+1} = y_n + h f(t_n, y_n)$; global error $O(h)$; energy grows by a factor $(1+h^2\omega^2)$ per step for an oscillator.
- **Euler–Cromer:** update $v$ first, then $x$ with the new $v$; symplectic, so energy stays bounded.
- **Midpoint/RK2:** $O(h^2)$. **RK4:** $O(h^4)$, the classic fixed-step workhorse. Measure convergence with a log-log plot of error versus $h$.
- `solve_ivp(f, (t0, t1), y0, rtol=..., atol=..., t_eval=..., dense_output=True, events=..., args=(...))`:
  - `sol.t`, `sol.y` (one **row** per variable), `sol.sol(t)` for dense output, `sol.t_events`, `sol.y_events`;
  - set `rtol`/`atol` yourself: the defaults are loose;
  - event functions return a quantity that crosses zero; set `.terminal` and `.direction`.
- Check simulations with **conservation laws** and **known limits** (vacuum range, small-angle period, Kepler's third law, nonrelativistic limit).
- **Stiff** equations need implicit methods: `method='Radau'`, `'BDF'`, or `'LSODA'`.
- New Python: tuple unpacking (`x, v = y`), `lambda`, passing functions as arguments, attributes on functions, list comprehensions.

## Further reading

- M. Newman, *Computational Physics*, Chapter 8 (Ordinary differential equations). Free chapters at [www-personal.umich.edu/~mejn/cp](http://www-personal.umich.edu/~mejn/cp/).
- SciPy documentation: [`scipy.integrate.solve_ivp`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html).
- J. VanderPlas, *Python Data Science Handbook* (free online) for more NumPy and Matplotlib.